# OpenOppsDB — Explorer

Featured Gradio app for the public snapshot. Tabs: jobs, companies, skills, and filters/plots. Route Ledger theme. Read-only, no credentials.

## Contents

- Setup (read-only `/kaggle/input`, `mode=ro&immutable=1`)
- Queries and charts for this kernel
- Links to the rest of the collection

## Collection

| Notebook | Kernel | What it is for |
| --- | --- | --- |
| **Starter** | [`wyattowalsh/openoppsdb-starter-notebook`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-starter-notebook) | Front door: tables, recent open jobs, first `%%sql` cells |
| **Explorer (featured)** | [`wyattowalsh/openoppsdb-explorer`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-explorer) | Gradio UI: jobs, companies, skills, filters/plots |
| **Advanced usage** | [`wyattowalsh/openoppsdb-advanced-usage`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-advanced-usage) | Joins, version history, company drill-down, Parquet |
| **SQL playground** | [`wyattowalsh/openoppsdb-sql-playground`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-sql-playground) | JupySQL studio: CTEs, DuckDB attach, Parquet scans |
| **Hiring market map** | [`wyattowalsh/openoppsdb-hiring-market-map`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-hiring-market-map) | Company, provider, location, and remote mix charts |
| **Skills radar** | [`wyattowalsh/openoppsdb-skills-radar`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-skills-radar) | Skill groups, keywords, and co-occurrence |
| **Snapshot health** | [`wyattowalsh/openoppsdb-snapshot-health`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-snapshot-health) | Coverage, freshness, sync runs, observation mix |


In [ ]:
%pip install -q gradio==6.26.0 plotly==7.0.0


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
ROUTE_LEDGER = {
    "pine": "#2f6f50",
    "paper": "#f7f1df",
    "brass": "#d99629",
    "ink": "#1d281f",
    "info": "#336d8f",
}

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
PARQUET_DIR = DATASET_DIR / "exports" / "parquet"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
import json
import gradio as gr

with sqlite3.connect(DB_URI, uri=True) as conn:
    jobs_df = pd.read_sql_query(
        """
        select j.id as job_id,
               coalesce(v.company, b.name, 'Unknown') as company,
               v.title, j.provider_id, j.status, v.remote,
               v.employment_type, j.last_seen_at, v.posting_url
        from jobs j
        join job_versions v on v.id = j.current_version_id
        left join boards b on b.key = j.board_key
        where j.status = 'open'
        """,
        conn,
    )
    skills_df = pd.read_sql_query(
        """
        select lower(s.name) as skill, count(distinct j.id) as open_roles
        from jobs j
        join job_versions v on v.id = j.current_version_id
        join job_version_skills s on s.job_version_id = v.id
        where j.status = 'open' and s.name is not null and s.name <> ''
        group by lower(s.name)
        order by open_roles desc, skill
        limit 50
        """,
        conn,
    )

companies = sorted(jobs_df["company"].dropna().unique().tolist()) if not jobs_df.empty else []
providers = sorted(jobs_df["provider_id"].dropna().unique().tolist()) if not jobs_df.empty else []
remotes = sorted(jobs_df["remote"].dropna().astype(str).unique().tolist()) if not jobs_df.empty else []

def _filter_jobs(company, provider, remote):
    frame = jobs_df.copy()
    if company:
        frame = frame[frame["company"] == company]
    if provider:
        frame = frame[frame["provider_id"] == provider]
    if remote:
        frame = frame[frame["remote"].astype(str) == remote]
    return frame.head(200)

def _company_dossier(company):
    if not company:
        return pd.DataFrame()
    return jobs_df[jobs_df["company"] == company].head(200)

def _jobs_plot(company, provider, remote):
    frame = _filter_jobs(company, provider, remote)
    if frame.empty:
        return px.bar(title="No rows")
    mix = frame.groupby("provider_id").size().reset_index(name="open_roles")
    fig = px.bar(
        mix,
        x="provider_id",
        y="open_roles",
        color_discrete_sequence=[ROUTE_LEDGER["pine"]],
        title="Open roles by provider",
    )
    fig.update_layout(paper_bgcolor=ROUTE_LEDGER["paper"], font_color=ROUTE_LEDGER["ink"])
    return fig

def _skills_plot():
    if skills_df.empty:
        return px.bar(title="No skills")
    fig = px.bar(
        skills_df.head(20).sort_values("open_roles"),
        x="open_roles",
        y="skill",
        orientation="h",
        color_discrete_sequence=[ROUTE_LEDGER["brass"]],
        title="Top skills",
    )
    fig.update_layout(paper_bgcolor=ROUTE_LEDGER["paper"], font_color=ROUTE_LEDGER["ink"])
    return fig

theme = gr.themes.Soft(primary_hue="green", secondary_hue="stone")
css = f"""
.gradio-container {{ background: {ROUTE_LEDGER["paper"]} !important; color: {ROUTE_LEDGER["ink"]}; }}
"""
with gr.Blocks(theme=theme, css=css, title="OpenOppsDB Explorer") as demo:
    gr.Markdown("# OpenOppsDB Explorer")
    with gr.Tab("jobs"):
        c1 = gr.Dropdown(choices=companies, label="company", value=None)
        p1 = gr.Dropdown(choices=providers, label="provider", value=None)
        r1 = gr.Dropdown(choices=remotes, label="remote", value=None)
        jobs_out = gr.Dataframe(value=jobs_df.head(50))
        c1.change(_filter_jobs, [c1, p1, r1], jobs_out)
        p1.change(_filter_jobs, [c1, p1, r1], jobs_out)
        r1.change(_filter_jobs, [c1, p1, r1], jobs_out)
    with gr.Tab("companies"):
        c2 = gr.Dropdown(choices=companies, label="company dossier", value=companies[0] if companies else None)
        dossier_out = gr.Dataframe(value=_company_dossier(companies[0]) if companies else pd.DataFrame())
        c2.change(_company_dossier, c2, dossier_out)
    with gr.Tab("skills"):
        gr.Dataframe(value=skills_df, wrap=True)
        gr.Plot(value=_skills_plot())
    with gr.Tab("filters/plots"):
        c3 = gr.Dropdown(choices=companies, label="company", value=None)
        p3 = gr.Dropdown(choices=providers, label="provider", value=None)
        r3 = gr.Dropdown(choices=remotes, label="remote", value=None)
        plot_out = gr.Plot(value=_jobs_plot(None, None, None))
        c3.change(_jobs_plot, [c3, p3, r3], plot_out)
        p3.change(_jobs_plot, [c3, p3, r3], plot_out)
        r3.change(_jobs_plot, [c3, p3, r3], plot_out)

summary = {
    "jobs": int(len(jobs_df)),
    "companies": int(len(companies)),
    "skills": int(len(skills_df)),
}
Path("/kaggle/working/openopps-explorer-summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

demo.launch()
